In [2]:
# Authenticate with NASA Earthdata
import earthaccess
earthaccess.login()

In [5]:
# Target point coordinates
# TN
# target_longitude = -68.86417
# target_latitude = -20.26568
# TS
target_longitude = -68.815688
target_latitude = -20.335576 

temporal_range = ("2026-01-17", "2026-11-01") # inicio 2012-01-17

ma4 = earthaccess.search_data(
    short_name="VNP43MA4",
    version="002",
    temporal=temporal_range,
    point=(target_longitude, target_latitude),
)
# print(f"Total VNP43MA4 granules found: {len(ma4)}")

ia4 = earthaccess.search_data(
    short_name="VNP43IA4",
    version="002",
    temporal=temporal_range,
    point=(target_longitude, target_latitude),
)
# print(f"Total VNP43IA4 granules found: {len(ia4)}")

/opt/miniconda3/envs/SMAP/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [6]:
import math
import os
import pyproj

# Create a local directory for sample files
sample_directory = "./sample_viirs"
os.makedirs(sample_directory, exist_ok=True)

# Download the latest granule for each product
latest_ma4_granule = earthaccess.download(ma4[-1:], local_path=sample_directory)[0]
latest_ia4_granule = earthaccess.download(ia4[-1:], local_path=sample_directory)[0]

# Standard MODIS/VIIRS sinusoidal projection parameters
earth_radius = 6371007.181
sinusoidal_crs_proj4 = f"+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R={earth_radius} +units=m +no_defs"

# Projections setup
transformer_to_sinu = pyproj.Transformer.from_crs("EPSG:4326", sinusoidal_crs_proj4, always_xy=True)
transformer_to_wgs84 = pyproj.Transformer.from_crs(sinusoidal_crs_proj4, "EPSG:4326", always_xy=True)

# Transform target geographic coordinates to sinusoidal projection meters
target_sinu_x, target_sinu_y = transformer_to_sinu.transform(target_longitude, target_latitude)

# Tile dimensions in sinusoidal projection (36 horizontal tiles, 18 vertical tiles globally)
tile_width_meters = (2.0 * math.pi * earth_radius) / 36.0
tile_height_meters = tile_width_meters

# Identify the tile horizontal (h) and vertical (v) indices for the target point
tile_h = int((target_sinu_x + math.pi * earth_radius) // tile_width_meters)
tile_v = int(((math.pi * earth_radius / 2.0) - target_sinu_y) // tile_height_meters)

# Compute tile upper-left corner coordinates in meters
tile_upper_left_x = -math.pi * earth_radius + tile_h * tile_width_meters
tile_upper_left_y = (math.pi * earth_radius / 2.0) - tile_v * tile_height_meters

print(f"Target point belongs to tile: h{tile_h:02d}v{tile_v:02d}")

# --- VNP43MA4 (1000m, 1200 x 1200 pixels) ---
ma4_num_cells = 1200
ma4_pixel_size = tile_width_meters / ma4_num_cells

ma4_col = int((target_sinu_x - tile_upper_left_x) // ma4_pixel_size)
ma4_row = int((tile_upper_left_y - target_sinu_y) // ma4_pixel_size)

ma4_pixel_min_x = tile_upper_left_x + ma4_col * ma4_pixel_size
ma4_pixel_max_x = ma4_pixel_min_x + ma4_pixel_size
ma4_pixel_max_y = tile_upper_left_y - ma4_row * ma4_pixel_size
ma4_pixel_min_y = ma4_pixel_max_y - ma4_pixel_size

ma4_nw_lon, ma4_nw_lat = transformer_to_wgs84.transform(ma4_pixel_min_x, ma4_pixel_max_y)
ma4_ne_lon, ma4_ne_lat = transformer_to_wgs84.transform(ma4_pixel_max_x, ma4_pixel_max_y)
ma4_se_lon, ma4_se_lat = transformer_to_wgs84.transform(ma4_pixel_max_x, ma4_pixel_min_y)
ma4_sw_lon, ma4_sw_lat = transformer_to_wgs84.transform(ma4_pixel_min_x, ma4_pixel_min_y)

ma4_pixel_polygon = [
    [ma4_nw_lat, ma4_nw_lon],
    [ma4_ne_lat, ma4_ne_lon],
    [ma4_se_lat, ma4_se_lon],
    [ma4_sw_lat, ma4_sw_lon],
]

# --- VNP43IA4 (500m, 2400 x 2400 pixels) ---
ia4_num_cells = 2400
ia4_pixel_size = tile_width_meters / ia4_num_cells

ia4_col = int((target_sinu_x - tile_upper_left_x) // ia4_pixel_size)
ia4_row = int((tile_upper_left_y - target_sinu_y) // ia4_pixel_size)

ia4_pixel_min_x = tile_upper_left_x + ia4_col * ia4_pixel_size
ia4_pixel_max_x = ia4_pixel_min_x + ia4_pixel_size
ia4_pixel_max_y = tile_upper_left_y - ia4_row * ia4_pixel_size
ia4_pixel_min_y = ia4_pixel_max_y - ia4_pixel_size

ia4_nw_lon, ia4_nw_lat = transformer_to_wgs84.transform(ia4_pixel_min_x, ia4_pixel_max_y)
ia4_ne_lon, ia4_ne_lat = transformer_to_wgs84.transform(ia4_pixel_max_x, ia4_pixel_max_y)
ia4_se_lon, ia4_se_lat = transformer_to_wgs84.transform(ia4_pixel_max_x, ia4_pixel_min_y)
ia4_sw_lon, ia4_sw_lat = transformer_to_wgs84.transform(ia4_pixel_min_x, ia4_pixel_min_y)

ia4_pixel_polygon = [
    [ia4_nw_lat, ia4_nw_lon],
    [ia4_ne_lat, ia4_ne_lon],
    [ia4_se_lat, ia4_se_lon],
    [ia4_sw_lat, ia4_sw_lon],
]

print(f"Target pixel position in MA4 (1000m): row={ma4_row}, col={ma4_col}")
print(f"Target pixel position in IA4 (500m):  row={ia4_row}, col={ia4_col}")

/opt/miniconda3/envs/SMAP/lib/python3.13/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Target point belongs to tile: h11v11
Target pixel position in MA4 (1000m): row=40, col=656
Target pixel position in IA4 (500m):  row=80, col=1313


In [ ]:
import folium
import ipywidgets
# requires ipywidgets

# Initialize the map centered at target coordinates
interactive_map = folium.Map(
    location=[target_latitude, target_longitude],
    zoom_start=14,
    tiles=None,
)

# Add satellite imagery layer (Esri World Imagery)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Esri Satellite",
    max_zoom=19,
).add_to(interactive_map)

# Add MA4 pixel footprint (~1000m resolution)
folium.Polygon(
    locations=ma4_pixel_polygon,
    color="blue",
    weight=2,
    fill=True,
    fill_color="blue",
    fill_opacity=0.15,
    tooltip="VNP43MA4 Pixel (~1000 m)",
    popup="VNP43MA4 Footprint",
).add_to(interactive_map)

# Add IA4 pixel footprint (~500m resolution)
folium.Polygon(
    locations=ia4_pixel_polygon,
    color="darkorange",
    weight=2,
    fill=True,
    fill_color="darkorange",
    fill_opacity=0.25,
    tooltip="VNP43IA4 Pixel (~500 m)",
    popup="VNP43IA4 Footprint",
).add_to(interactive_map)

# Add target location marker
folium.CircleMarker(
    location=[target_latitude, target_longitude],
    radius=5,
    color="red",
    fill=True,
    fill_color="red",
    fill_opacity=1.0,
    tooltip="Target Point",
    popup=f"Lat: {target_latitude}, Lon: {target_longitude}",
).add_to(interactive_map)

folium.LayerControl().add_to(interactive_map)
interactive_map

Solución por si no aparece el mapa:
https://stackoverflow.com/questions/63245563/make-this-notebook-trusted-to-load-map-file-trust-notebook